In [1]:
import sys
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
cwd = Path.cwd()
project_root = cwd.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
    print('Done!')

Done!


In [ ]:
from src.get_data_sample import get_sample_data, DATA_ENDPOINT
from src.utils.data_check_functions import api_data_return_check

In [4]:
fixture = pd.read_csv(
    project_root / 'data' / 'england_fixtures_pre_euro_2024.csv',
    parse_dates=['match_date_london']
)

In [5]:
fixture['scheduled_kickoff_utc'] = pd.to_datetime(
    fixture['scheduled_kickoff_utc'], utc=True
)

fixture['scheduled_kickoff_london'] = (
    fixture['scheduled_kickoff_utc'].dt.tz_convert('Europe/London')
)

In [6]:
fixture.head()

,match_id,match_date_london,opponent,england_home_away,competition,scheduled_kickoff_clock,source_timezone,scheduled_kickoff_london,scheduled_kickoff_utc,timing_source_url,inventory_source_url,timing_status,notes
0,ENG_20230323,2023-03-23,Italy,away,UEFA EURO 2024 qualifying,19:45,GMT,2023-03-23 19:45:00+00:00,2023-03-23 19:45:00+00:00,https://www.englandfootball.com/articles/2023/...,https://www.englandfootball.com/england/mens-s...,scheduled_time_verified,NaN
1,ENG_20230326,2023-03-26,Ukraine,home,UEFA EURO 2024 qualifying,17:00,BST,2023-03-26 17:00:00+01:00,2023-03-26 16:00:00+00:00,https://www.englandfootball.com/articles/2023/...,https://www.englandfootball.com/england/mens-s...,scheduled_time_verified,Match-centre preview labels 17:00 GMT; use exp...
2,ENG_20230616,2023-06-16,Malta,away,UEFA EURO 2024 qualifying,19:45,BST,2023-06-16 19:45:00+01:00,2023-06-16 18:45:00+00:00,https://www.englandfootball.com/articles/2023/...,https://www.englandfootball.com/england/mens-s...,scheduled_time_verified,NaN
3,ENG_20230619,2023-06-19,North Macedonia,home,UEFA EURO 2024 qualifying,19:45,BST,2023-06-19 19:45:00+01:00,2023-06-19 18:45:00+00:00,https://www.englandfootball.com/articles/2023/...,https://www.englandfootball.com/england/mens-s...,scheduled_time_verified,Match-centre preview labels 19:45 GMT; advance...
4,ENG_20230909,2023-09-09,Ukraine,away,UEFA EURO 2024 qualifying,17:00,BST,2023-09-09 17:00:00+01:00,2023-09-09 16:00:00+00:00,https://www.englandfootball.com/articles/2023/...,https://www.englandfootball.com/england/mens-s...,scheduled_time_verified,Played in Poland; listed time is British time ...


In [7]:
fixture.shape

(12, 13)

In [8]:
fixture.columns

Index(['match_id', 'match_date_london', 'opponent', 'england_home_away',
       'competition', 'scheduled_kickoff_clock', 'source_timezone',
       'scheduled_kickoff_london', 'scheduled_kickoff_utc',
       'timing_source_url', 'inventory_source_url', 'timing_status', 'notes'],
      dtype='str')

In [9]:
fixture[['match_id', 'match_date_london', 'opponent', 'scheduled_kickoff_london', 'scheduled_kickoff_utc']]

,match_id,match_date_london,opponent,scheduled_kickoff_london,scheduled_kickoff_utc
0,ENG_20230323,2023-03-23,Italy,2023-03-23 19:45:00+00:00,2023-03-23 19:45:00+00:00
1,ENG_20230326,2023-03-26,Ukraine,2023-03-26 17:00:00+01:00,2023-03-26 16:00:00+00:00
2,ENG_20230616,2023-06-16,Malta,2023-06-16 19:45:00+01:00,2023-06-16 18:45:00+00:00
3,ENG_20230619,2023-06-19,North Macedonia,2023-06-19 19:45:00+01:00,2023-06-19 18:45:00+00:00
4,ENG_20230909,2023-09-09,Ukraine,2023-09-09 17:00:00+01:00,2023-09-09 16:00:00+00:00
5,ENG_20230912,2023-09-12,Scotland,2023-09-12 19:45:00+01:00,2023-09-12 18:45:00+00:00
6,ENG_20231013,2023-10-13,Australia,2023-10-13 19:45:00+01:00,2023-10-13 18:45:00+00:00
7,ENG_20231017,2023-10-17,Italy,2023-10-17 19:45:00+01:00,2023-10-17 18:45:00+00:00
8,ENG_20231117,2023-11-17,Malta,2023-11-17 19:45:00+00:00,2023-11-17 19:45:00+00:00
9,ENG_20231120,2023-11-20,North Macedonia,2023-11-20 19:45:00+00:00,2023-11-20 19:45:00+00:00


In [10]:
fixture.info()

<class 'pandas.DataFrame'>
RangeIndex: 12 entries, 0 to 11
Data columns (total 13 columns):
 #   Column                    Non-Null Count  Dtype                        
---  ------                    --------------  -----                        
 0   match_id                  12 non-null     str                          
 1   match_date_london         12 non-null     datetime64[us]               
 2   opponent                  12 non-null     str                          
 3   england_home_away         12 non-null     str                          
 4   competition               12 non-null     str                          
 5   scheduled_kickoff_clock   12 non-null     str                          
 6   source_timezone           12 non-null     str                          
 7   scheduled_kickoff_london  12 non-null     datetime64[us, Europe/London]
 8   scheduled_kickoff_utc     12 non-null     datetime64[us, UTC]          
 9   timing_source_url         12 non-null     str           

In [11]:
len(fixture['match_id'].unique())

12

In [12]:
fixture.iloc[0]['match_date_london']

Timestamp('2023-03-23 00:00:00')

In [13]:
start_time = fixture.iloc[0]['match_date_london']
end_time = start_time + pd.Timedelta(days=1)

In [14]:
start_time, end_time

(Timestamp('2023-03-23 00:00:00'), Timestamp('2023-03-24 00:00:00'))

In [15]:
start_london = start_time.tz_localize('Europe/London')
end_london = end_time.tz_localize('Europe/London')

In [16]:
start_utc = start_london.tz_convert('UTC')
end_utc = end_london.tz_convert('UTC')

In [17]:
start_utc, end_utc

(Timestamp('2023-03-23 00:00:00+0000', tz='UTC'),
 Timestamp('2023-03-24 00:00:00+0000', tz='UTC'))

In [18]:
print(f'Match_id: {fixture.iloc[0]['match_id']}')
print(f'Local Match date start day time: {start_london}')
print(f'Local Match date end day time: {end_london}')
print(f'UTC Match date start day time: {start_utc}')
print(f'UTC Match date end day time: {end_utc}')

Match_id: ENG_20230323
Local Match date start day time: 2023-03-23 00:00:00+00:00
Local Match date end day time: 2023-03-24 00:00:00+00:00
UTC Match date start day time: 2023-03-23 00:00:00+00:00
UTC Match date end day time: 2023-03-24 00:00:00+00:00


In [19]:
params = {
    'from': start_utc.isoformat(),
    'to': end_utc.isoformat(),
    'format': 'json'
}

test_data = get_sample_data(
    endpoint=DATA_ENDPOINT,
    params=params
)

In [20]:
test_data, checks = api_data_return_check(
    data=test_data,
    start_utc=start_utc,
    end_utc=end_utc
)

Converting the data into a dataframe...
Sucessfully converted!
Converting the startTime column in a UTC time stamp...
Successfully converted!
Removing the last index and sorting...
Checking unexpected timestamps...


In [21]:
test_data.head()

,recordType,startTime,demand
0,VD,2023-03-23 00:00:00+00:00,24635
1,VD,2023-03-23 00:05:00+00:00,24323
2,VD,2023-03-23 00:10:00+00:00,24254
3,VD,2023-03-23 00:15:00+00:00,24162
4,VD,2023-03-23 00:20:00+00:00,24091


In [22]:
checks

{'null_timestamp_count_before_filter': np.int64(0),
 'expected_no_of_rows': 288,
 'actual_no_of_rows': 288,
 'missing_timestamp_count': 0,
 'unexpected_timestamp_count': 0,
 'null_demand_count': 0,
 'demand_is_numeric': True,
 'record_type_counts': {'VD': 288},
 'gaps_five_mins': np.True_,
 'duplicate_timestamp_count': np.int64(0)}

In [29]:
results = []
checked_data_by_match = {}

for _, match in fixture.iterrows():
    match_id = match['match_id']
    match_date = match['match_date_london']
    summary = {
        'match_id': match_id,
        'match_date_london': match_date,
    }

    try:
        start_london = match_date.tz_localize('Europe/London')
        end_london = (match_date + pd.Timedelta(days=1)).tz_localize('Europe/London')
        start_utc = start_london.tz_convert('UTC')
        end_utc = end_london.tz_convert('UTC')

        params = {
            'from': start_utc.isoformat(),
            'to': end_utc.isoformat(),
            'format': 'json',
        }
        raw_data = get_sample_data(endpoint=DATA_ENDPOINT, params=params)
        checked_data, checks = api_data_return_check(
            data=raw_data,
            start_utc=start_utc,
            end_utc=end_utc,
        )

        checked_data_by_match[match_id] = checked_data
        summary.update(checks)
        summary.update(status='success', error=None)
    except Exception as exc:
        summary.update(
            status='failed',
            error=f'{type(exc).__name__}: {exc}',
        )

    results.append(summary)

coverage_summary = pd.DataFrame(results)


Converting the data into a dataframe...
Sucessfully converted!
Converting the startTime column in a UTC time stamp...
Successfully converted!
Removing the last index and sorting...
Checking unexpected timestamps...
Converting the data into a dataframe...
Sucessfully converted!
Converting the startTime column in a UTC time stamp...
Successfully converted!
Removing the last index and sorting...
Checking unexpected timestamps...
Converting the data into a dataframe...
Sucessfully converted!
Converting the startTime column in a UTC time stamp...
Successfully converted!
Removing the last index and sorting...
Checking unexpected timestamps...
Converting the data into a dataframe...
Sucessfully converted!
Converting the startTime column in a UTC time stamp...
Successfully converted!
Removing the last index and sorting...
Checking unexpected timestamps...
Converting the data into a dataframe...
Sucessfully converted!
Converting the startTime column in a UTC time stamp...
Successfully converted

In [30]:
coverage_summary

,match_id,match_date_london,null_timestamp_count_before_filter,expected_no_of_rows,actual_no_of_rows,missing_timestamp_count,unexpected_timestamp_count,null_demand_count,demand_is_numeric,record_type_counts,gaps_five_mins,duplicate_timestamp_count,status,error
0,ENG_20230323,2023-03-23,0,288,288,0,0,0,True,{'VD': 288},True,0,success,None
1,ENG_20230326,2023-03-26,0,276,276,0,0,0,True,{'VD': 276},True,0,success,None
2,ENG_20230616,2023-06-16,0,288,288,0,0,0,True,{'VD': 288},True,0,success,None
3,ENG_20230619,2023-06-19,0,288,288,0,0,0,True,{'VD': 288},True,0,success,None
4,ENG_20230909,2023-09-09,0,288,288,0,0,0,True,{'VD': 288},True,0,success,None
5,ENG_20230912,2023-09-12,0,288,288,0,0,0,True,{'VD': 288},True,0,success,None
6,ENG_20231013,2023-10-13,0,288,288,0,0,0,True,{'VD': 288},True,0,success,None
7,ENG_20231017,2023-10-17,0,288,288,0,0,0,True,{'VD': 288},True,0,success,None
8,ENG_20231117,2023-11-17,0,288,288,0,0,0,True,{'VD': 288},True,0,success,None
9,ENG_20231120,2023-11-20,0,288,288,0,0,0,True,{'VD': 288},True,0,success,None


In [31]:
results

[{'match_id': 'ENG_20230323',
  'match_date_london': Timestamp('2023-03-23 00:00:00'),
  'null_timestamp_count_before_filter': np.int64(0),
  'expected_no_of_rows': 288,
  'actual_no_of_rows': 288,
  'missing_timestamp_count': 0,
  'unexpected_timestamp_count': 0,
  'null_demand_count': 0,
  'demand_is_numeric': True,
  'record_type_counts': {'VD': 288},
  'gaps_five_mins': np.True_,
  'duplicate_timestamp_count': np.int64(0),
  'status': 'success',
  'error': None},
 {'match_id': 'ENG_20230326',
  'match_date_london': Timestamp('2023-03-26 00:00:00'),
  'null_timestamp_count_before_filter': np.int64(0),
  'expected_no_of_rows': 276,
  'actual_no_of_rows': 276,
  'missing_timestamp_count': 0,
  'unexpected_timestamp_count': 0,
  'null_demand_count': 0,
  'demand_is_numeric': True,
  'record_type_counts': {'VD': 276},
  'gaps_five_mins': np.True_,
  'duplicate_timestamp_count': np.int64(0),
  'status': 'success',
  'error': None},
 {'match_id': 'ENG_20230616',
  'match_date_london': Tim

In [34]:
assert len(coverage_summary['match_id'].unique()) == len(coverage_summary)

In [35]:
coverage_summary.loc[coverage_summary['status'] == 'failed']

,match_id,match_date_london,null_timestamp_count_before_filter,expected_no_of_rows,actual_no_of_rows,missing_timestamp_count,unexpected_timestamp_count,null_demand_count,demand_is_numeric,record_type_counts,gaps_five_mins,duplicate_timestamp_count,status,error


In [36]:
coverage_summary.shape

(12, 14)

In [37]:
checked_data_by_match

{'ENG_20230323':     recordType                 startTime  demand
 0           VD 2023-03-23 00:00:00+00:00   24635
 1           VD 2023-03-23 00:05:00+00:00   24323
 2           VD 2023-03-23 00:10:00+00:00   24254
 3           VD 2023-03-23 00:15:00+00:00   24162
 4           VD 2023-03-23 00:20:00+00:00   24091
 ..         ...                       ...     ...
 283         VD 2023-03-23 23:35:00+00:00   24331
 284         VD 2023-03-23 23:40:00+00:00   24158
 285         VD 2023-03-23 23:45:00+00:00   23920
 286         VD 2023-03-23 23:50:00+00:00   23673
 287         VD 2023-03-23 23:55:00+00:00   23751
 
 [288 rows x 3 columns],
 'ENG_20230326':     recordType                 startTime  demand
 0           VD 2023-03-26 00:00:00+00:00   23186
 1           VD 2023-03-26 00:05:00+00:00   23300
 2           VD 2023-03-26 00:10:00+00:00   23230
 3           VD 2023-03-26 00:15:00+00:00   23167
 4           VD 2023-03-26 00:20:00+00:00   23191
 ..         ...                       ...

In [39]:
coverage_summary.to_csv(
    project_root / 'data' / 'processed' / 'fixture_coverage_summary.csv',
    index=False
)

## Fixture coverage audit conclusion

All 12 historical fixture days were retrieved and passed the implemented structural checks. Actual row counts matched the expected five-minute timelines, with no missing, unexpected, or duplicate timestamps and no null timestamps or demand values. Demand was numeric, all records were `VD`, and all consecutive gaps were five minutes.

Eleven days contained 288 observations. The England–Ukraine fixture on 26 March 2023 contained 276 observations because the London clock moved forward that day, making the local calendar day 23 hours long. Requests used London-midnight boundaries converted to UTC and excluded the following midnight.

These results establish coverage for the 12 tested fixture days, not continuous coverage of the full training period or suitability for causal inference. The snapshot-versus-average convention and precise timestamp meaning remain unresolved.

The audit summary is saved at `data/processed/fixture_coverage_summary.csv`. Checked demand DataFrames are retained in the notebook session by match ID; they have not been established as persisted data files by this audit.
